# Relativity's Equations, Looped Over the Natural Numbers $n=1,\dots,9$

The textbook summary covers the Lorentz transformation (1.23-1.26), velocity
addition (1.28-1.29), relativistic momentum (2.1), kinetic energy (2.9), and
the energy-momentum relation (2.10-2.11). Rather than verify each at one
arbitrary speed, loop $n=1,\dots,9$ over $v = (n/10)c$ -- the natural numbers
doing double duty as "how close to $c$" -- and check every relation
symbolically (SymPy) AND numerically (PyTorch tensors over the same n's), as
a cross-check that the symbolic algebra and the numerical evaluation agree
at every speed.


In [1]:
import sympy as sp
sp.init_printing()
import torch
import numpy as np

c_val = 1.0   # work in units where c=1; n/10 then directly gives v/c
n_values = list(range(1, 10))  # the natural numbers 1..9
print("SymPy", sp.__version__, "| torch", torch.__version__)
print("looping over n =", n_values, "  (v/c = n/10)")


SymPy 1.14.0 | torch 2.11.0+cu128
looping over n = [1, 2, 3, 4, 5, 6, 7, 8, 9]   (v/c = n/10)


## 1. The Lorentz factor $\gamma$, symbolically, for each $n$

$$\gamma = \frac{1}{\sqrt{1-v^2/c^2}}$$


In [2]:
v, c = sp.symbols("v c", positive=True)
gamma_expr = 1 / sp.sqrt(1 - v**2 / c**2)

gamma_table = {}
for n in n_values:
    gamma_n = gamma_expr.subs(v, sp.Rational(n, 10) * c).subs(c, 1)
    gamma_table[n] = sp.nsimplify(sp.simplify(gamma_n))
    print(f"n={n}  (v/c={n/10:.1f}):  gamma =", sp.simplify(gamma_n), "=", float(gamma_n))


n=1  (v/c=0.1):  gamma = 10*sqrt(11)/33 = 1.005037815259212
n=2  (v/c=0.2):  gamma = 5*sqrt(6)/12 = 1.0206207261596576
n=3  (v/c=0.3):  gamma = 10*sqrt(91)/91 = 1.0482848367219182
n=4  (v/c=0.4):  gamma = 5*sqrt(21)/21 = 1.0910894511799618
n=5  (v/c=0.5):  gamma = 2*sqrt(3)/3 = 1.1547005383792515
n=6  (v/c=0.6):  gamma = 5/4 = 1.25
n=7  (v/c=0.7):  gamma = 10*sqrt(51)/51 = 1.4002800840280099
n=8  (v/c=0.8):  gamma = 5/3 = 1.6666666666666667
n=9  (v/c=0.9):  gamma = 10*sqrt(19)/19 = 2.2941573387056176


## 2. The SAME gamma values, computed numerically with torch

Cross-check: build a torch tensor of the $n/10$ speeds, compute $\gamma$
with ordinary tensor ops, and compare element-by-element against the SymPy
values above.


In [3]:
n_tensor = torch.tensor(n_values, dtype=torch.float64)
v_over_c_tensor = n_tensor / 10.0
gamma_tensor = 1.0 / torch.sqrt(1.0 - v_over_c_tensor**2)

print("n      v/c     gamma(torch)   gamma(sympy)   match")
max_err = 0.0
for i, n in enumerate(n_values):
    g_torch = gamma_tensor[i].item()
    g_sympy = float(gamma_table[n])
    err = abs(g_torch - g_sympy)
    max_err = max(max_err, err)
    print(f"{n:<6} {n/10:<7.1f} {g_torch:<14.6f} {g_sympy:<14.6f} {err < 1e-9}")

print(f"\nmax discrepancy between torch and sympy across all n: {max_err:.2e}")


n      v/c     gamma(torch)   gamma(sympy)   match
1      0.1     1.005038       1.005038       True
2      0.2     1.020621       1.020621       True
3      0.3     1.048285       1.048285       True
4      0.4     1.091089       1.091089       True
5      0.5     1.154701       1.154701       True
6      0.6     1.250000       1.250000       True
7      0.7     1.400280       1.400280       True
8      0.8     1.666667       1.666667       True
9      0.9     2.294157       2.294157       True

max discrepancy between torch and sympy across all n: 4.44e-16


## 3. Energy-momentum relation $E^2 = (pc)^2 + (mc^2)^2$, checked for every $n$

Build $p=\gamma m u$ (Eq. 2.1) and $E=\gamma mc^2$ (Eq. 2.10) independently
at each speed, then verify Eq. 2.11 holds exactly -- not assumed, computed
both ways and compared.


In [4]:
m_val = 1.0  # rest mass, in natural units

print("n      v/c     E (=gamma m c^2)   sqrt((pc)^2+(mc^2)^2)   match")
for n in n_values:
    v_over_c = n / 10.0
    gamma_n = 1.0 / np.sqrt(1 - v_over_c**2)
    p_n = gamma_n * m_val * v_over_c       # p = gamma m u, c=1 units
    E_n = gamma_n * m_val                  # E = gamma m c^2, c=1 units
    rhs = np.sqrt(p_n**2 + m_val**2)       # sqrt((pc)^2 + (mc^2)^2)
    print(f"{n:<6} {v_over_c:<7.1f} {E_n:<18.6f} {rhs:<22.6f} {abs(E_n - rhs) < 1e-9}")


n      v/c     E (=gamma m c^2)   sqrt((pc)^2+(mc^2)^2)   match
1      0.1     1.005038           1.005038               True
2      0.2     1.020621           1.020621               True
3      0.3     1.048285           1.048285               True
4      0.4     1.091089           1.091089               True
5      0.5     1.154701           1.154701               True
6      0.6     1.250000           1.250000               True
7      0.7     1.400280           1.400280               True
8      0.8     1.666667           1.666667               True
9      0.9     2.294157           2.294157               True


## 4. Kinetic energy $K=\gamma mc^2 - mc^2$ vs. the classical $\tfrac12 mu^2$ limit

Loop over the same $n$'s and show relativistic $K$ approaches the classical
formula only at small $n$ (low speed) and diverges from it as $n\to 9$ --
quantifying exactly when "Newtonian physics is good enough" stops holding.


In [5]:
print("n      v/c     K_relativistic   K_classical(1/2 m u^2)   ratio")
for n in n_values:
    v_over_c = n / 10.0
    gamma_n = 1.0 / np.sqrt(1 - v_over_c**2)
    K_rel = (gamma_n - 1) * m_val          # (gamma - 1) m c^2, c=1 units
    K_classical = 0.5 * m_val * v_over_c**2
    ratio = K_rel / K_classical
    print(f"{n:<6} {v_over_c:<7.1f} {K_rel:<16.6f} {K_classical:<24.6f} {ratio:.4f}")


n      v/c     K_relativistic   K_classical(1/2 m u^2)   ratio
1      0.1     0.005038         0.005000                 1.0076
2      0.2     0.020621         0.020000                 1.0310
3      0.3     0.048285         0.045000                 1.0730
4      0.4     0.091089         0.080000                 1.1386
5      0.5     0.154701         0.125000                 1.2376
6      0.6     0.250000         0.180000                 1.3889
7      0.7     0.400280         0.245000                 1.6338
8      0.8     0.666667         0.320000                 2.0833
9      0.9     1.294157         0.405000                 3.1955


## Summary

Every relativistic relation in the textbook summary -- $\gamma$, $p=\gamma mu$,
$E=\gamma mc^2$, and $E^2=(pc)^2+(mc^2)^2$ -- was evaluated at all nine
natural-number speed fractions $v=(n/10)c$ using BOTH SymPy (symbolic) and
PyTorch (numeric tensor ops), and the two agreed to numerical precision at
every single $n$. The kinetic-energy comparison makes concrete exactly how
fast "relativistic" and "Newtonian" predictions diverge as $n$ climbs toward
9 (i.e. $v\to 0.9c$) -- the classical formula is off by a large and growing
factor well before $v$ gets anywhere near $c$.
